# BTL02 T0402 ? EXP-11 Class Imbalance (Colab GPU)
Drive local-first sync; Secrets GITHUB_TOKEN/KAGGLE_API_TOKEN; per-class report l?u th?nh MLflow artifact.

In [ ]:
import os, shutil, subprocess, sys, tempfile, time
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError('B?t Colab GPU tr??c khi ch?y T0402.')
from google.colab import drive, userdata
drive.mount('/content/drive')
BRANCH = 'exp/BTL02_T0402_ablation_class_imbalance'
REPO_URL = 'https://github.com/Duy-DS/Deep_Learning_Group.git'
REPO_DIR = Path('/content/Deep_Learning_Group')
PROJECT_DIR = REPO_DIR / 'BTL02'
DRIVE_ROOT = Path('/content/drive/MyDrive/BTL02/T0402_Imbalance')
DRIVE_MLRUNS, DRIVE_CHECKPOINTS, DRIVE_LOGS = DRIVE_ROOT/'mlruns', DRIVE_ROOT/'checkpoints', DRIVE_ROOT/'logs'
LOCAL_MLRUNS, LOCAL_CHECKPOINTS, LOCAL_LOGS = Path('/content/mlruns_t0402'), Path('/content/checkpoints_t0402'), Path('/content/logs_t0402')
for folder in (DRIVE_ROOT, DRIVE_MLRUNS, DRIVE_CHECKPOINTS, DRIVE_LOGS, LOCAL_MLRUNS, LOCAL_CHECKPOINTS, LOCAL_LOGS):
    folder.mkdir(parents=True, exist_ok=True)
(LOCAL_MLRUNS/'.trash').mkdir(parents=True, exist_ok=True)
os.environ['MLFLOW_TRACKING_URI'] = LOCAL_MLRUNS.as_uri()
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))


In [ ]:
def sync_tree(source, destination, stable_age_seconds=0):
    source, destination = Path(source), Path(destination)
    if not source.exists():
        return
    destination.mkdir(parents=True, exist_ok=True)
    for root, dirs, files in os.walk(source):
        current = Path(root)
        dirs[:] = [name for name in dirs if name != '.trash']
        target_root = destination / current.relative_to(source)
        target_root.mkdir(parents=True, exist_ok=True)
        for filename in files:
            item = current / filename
            if item.is_symlink() or filename.endswith(('.tmp', '.part', '.sync-tmp')):
                continue
            stat = item.stat()
            if stable_age_seconds and time.time() - stat.st_mtime < stable_age_seconds:
                continue
            target = target_root / filename
            if target.exists() and target.stat().st_size == stat.st_size and target.stat().st_mtime_ns >= stat.st_mtime_ns:
                continue
            temporary = target.with_name(target.name + '.sync-tmp')
            shutil.copy2(item, temporary)
            if item.stat().st_size != stat.st_size or temporary.stat().st_size != stat.st_size:
                temporary.unlink(missing_ok=True)
                continue
            os.replace(temporary, target)
sync_tree(DRIVE_MLRUNS, LOCAL_MLRUNS)
sync_tree(DRIVE_CHECKPOINTS, LOCAL_CHECKPOINTS)
(LOCAL_MLRUNS/'.trash').mkdir(parents=True, exist_ok=True)
token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
git_env = os.environ.copy()
askpass_path = None
if token:
    fd, askpass_path = tempfile.mkstemp(prefix='t0402-git-askpass-')
    askpass_source = '#!/bin/sh\ncase "$1" in *Username*) printf "%s\\n" x-access-token ;; *) printf "%s\\n" "$T0402_GITHUB_TOKEN" ;; esac\n'
    with os.fdopen(fd, 'w', encoding='utf-8') as stream:
        stream.write(askpass_source)
    os.chmod(askpass_path, 0o700)
    git_env.update({'GIT_ASKPASS': askpass_path, 'GIT_TERMINAL_PROMPT':'0', 'T0402_GITHUB_TOKEN':token})
try:
    if REPO_DIR.exists():
        subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True,env=git_env)
        subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True,env=git_env)
        subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True,env=git_env)
    else:
        subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True,env=git_env)
finally:
    if askpass_path and os.path.exists(askpass_path):
        os.remove(askpass_path)
    git_env.pop('T0402_GITHUB_TOKEN',None)
    token = None
os.chdir(PROJECT_DIR)
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip() == BRANCH


In [ ]:
requirements = Path('requirements.txt').read_text(encoding='utf-8').splitlines()
packages = [line for line in requirements if line.strip() and not line.lstrip().startswith('#') and line.split('=',1)[0].strip().lower() not in {'torch','torchvision'}]
subprocess.run([sys.executable,'-m','pip','install','-q',*packages],check=True)
from src.data.downloader import FER2013Downloader
raw_csv = Path('data/raw/fer2013.csv')
raw_csv.parent.mkdir(parents=True,exist_ok=True)
drive_csv = DRIVE_ROOT/'fer2013.csv'
if not raw_csv.is_file() and drive_csv.is_file():
    shutil.copy2(drive_csv,raw_csv)
if not raw_csv.is_file():
    token = None
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except Exception:
        pass
    if not token:
        raise RuntimeError('C?n Colab Secret KAGGLE_API_TOKEN ho?c FER2013 CSV trong Drive.')
    os.environ['KAGGLE_API_TOKEN'] = token
    token = None
    try:
        subprocess.run([sys.executable,'-m','src.data.downloader'],check=True)
    finally:
        os.environ.pop('KAGGLE_API_TOKEN',None)
counts = FER2013Downloader.validate_csv(raw_csv)
assert counts == {'Training':28709,'PublicTest':3589,'PrivateTest':3589}, counts
if not drive_csv.exists():
    shutil.copy2(raw_csv,drive_csv)
outputs = Path('outputs')
outputs.mkdir(exist_ok=True)
for name,target in (('checkpoints',LOCAL_CHECKPOINTS),('logs',LOCAL_LOGS)):
    link = outputs/name
    if link.is_symlink() and link.resolve() != target.resolve():
        raise RuntimeError(f'Output symlink sai: {link}')
    if link.exists() and not link.is_symlink():
        if any(item.name != '.gitkeep' for item in link.iterdir()):
            raise RuntimeError(f'Output c? d? li?u; kh?ng ghi ??: {link}')
        for item in link.iterdir():
            item.unlink()
        link.rmdir()
    if not link.exists():
        link.symlink_to(target,target_is_directory=True)
print('FER2013 splits:',counts)


In [ ]:
import pandas as pd, yaml
from mlflow.tracking import MlflowClient
from src.train.trainer import Trainer
from src.train.mlflow_logger import MLflowLogger
CONFIG_FILES = ['exp11_loss_ce.yaml','exp11_loss_weighted_ce.yaml','exp11_loss_focal.yaml']
configs = []
for filename in CONFIG_FILES:
    config = yaml.safe_load((Path('experiments/ablation')/filename).read_text(encoding='utf-8'))
    config['run_name'] = Trainer._format_run_name(
        exp_id=config['exp_id'], architecture=config['architecture'], optimizer=config['optimizer'],
        pretrained=config['pretrained'], image_size=config['image_size'],
        augmentation_mode=config['augmentation_mode'], loss_name=config['loss']['name'])
    configs.append(config)
client = MlflowClient(tracking_uri=os.environ['MLFLOW_TRACKING_URI'])
experiment = client.get_experiment_by_name('BTL02_FER2013_PyTorch')
finished = set()
if experiment:
    for run in client.search_runs([experiment.experiment_id],max_results=10000):
        if run.info.status == 'FINISHED':
            finished.add(run.data.tags.get('mlflow.runName',''))
manifest = pd.DataFrame([{'config':name,'loss':item['loss']['name'],'run_name':item['run_name'],
                          'status':'DONE' if item['run_name'] in finished else 'PENDING'}
                         for name,item in zip(CONFIG_FILES,configs)])
display(manifest)
assert len(MLflowLogger.CSV_COLUMNS) == 24


In [ ]:
CSV_NAME = 't0402_ablation_imbalance_24col.csv'
local_csv = Path('outputs/logs')/CSV_NAME
drive_csv = DRIVE_LOGS/CSV_NAME
def sync_and_export():
    logger = MLflowLogger()
    raw = LOCAL_LOGS/'t0402_all_runs_export.csv'
    logger.export_summary_csv(str(raw))
    frame = pd.read_csv(raw,encoding='utf-8-sig',keep_default_na=False)
    assert frame.columns.tolist() == MLflowLogger.CSV_COLUMNS
    names = {item['run_name'] for item in configs}
    frame = frame[frame['Name'].isin(names) & frame['Status'].astype(str).str.upper().eq('FINISHED')]
    temporary = local_csv.with_name(local_csv.name+'.sync-tmp')
    frame.to_csv(temporary,index=False,encoding='utf-8-sig')
    os.replace(temporary,local_csv)
    drive_tmp = drive_csv.with_name(drive_csv.name+'.sync-tmp')
    shutil.copy2(local_csv,drive_tmp)
    os.replace(drive_tmp,drive_csv)
    sync_tree(LOCAL_MLRUNS,DRIVE_MLRUNS,stable_age_seconds=3)
    sync_tree(LOCAL_CHECKPOINTS,DRIVE_CHECKPOINTS)
    sync_tree(LOCAL_LOGS,DRIVE_LOGS)
    raw.unlink(missing_ok=True)
    return frame
failed = []
for index,config in enumerate(configs):
    if config['run_name'] in finished:
        print('[AUTO-SKIP FINISHED]',config['run_name'])
        sync_and_export()
        continue
    result = subprocess.run([sys.executable,'-u','-m','src.train.trainer','--config',
                             str(Path('experiments/ablation')/CONFIG_FILES[index])],cwd=PROJECT_DIR)
    if result.returncode:
        failed.append((config['run_name'],result.returncode))
    sync_and_export()
final = sync_and_export()
assert final.columns.tolist() == MLflowLogger.CSV_COLUMNS
print('FINISHED rows:',len(final),'/ 3; failures:',failed,'CSV:',drive_csv)
print('PrivateTest precision/recall/F1 by class are saved as JSON artifacts in each MLflow run.')

display(final[['Name','Run ID','Status','metrics.best_epoch','metrics.train_loss','metrics.train_acc','metrics.best_val_acc','metrics.private_test_acc','metrics.macro_f1']])
